# 原时间有限支持 KL 调参

树支持域为 `[0, upper_bound]`，`upper_bound = 1.1 * max_observation`。
每一折只用该折拟合数据的有限 L/R 端点计算上界，最终拟合只用前80%训练数据。
不使用非线性时间变换。保留固定 rho、adaptive KL 和匹配架构固定 rho 三组。
旧归一化实验及结果保持独立；新结果使用 `kl_cv_server_raw_time_v1`。


In [ ]:
from pathlib import Path
import sys
import torch
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'pycode'/'unified_nll').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Run inside PolyaPFN')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pycode.unified_nll.experiment import ExperimentConfig
from pycode.unified_nll.cv_search import CVSearchConfig
from pycode.unified_nll.train import TrainingConfig
from pycode.unified_nll.tuning_model import Architecture
from pycode.unified_nll.nll_cv_train import NLLCVOptimization
from pycode.unified_nll.raw_time.search import tune_nll_cv, evaluate_nll_cv, METHODS
from pycode.unified_nll.diagnostic_experiment import expanded_settings
from pycode.unified_nll.data_compatibility import verify_generator

OUTPUT = ROOT/'pycode'/'unified_nll'/'results'/'kl_cv_server_raw_time_v1'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CPU_THREADS = 4
CONFIG = ExperimentConfig(
    scenarios=('ph', 'lognormal', 'po', 'multimodal'),
    censor_types=('exact', 'right', 'interval', 'current'),
    censor_qs=(0.2, 0.5), n_samples=5000, grid_size=256,
    data_seed=123, seeds=tuple(range(10)),
    train_fraction=0.6, validation_fraction=0.2, resample_data=False,
)
SEARCH = CVSearchConfig(
    trials=64, initial_trials=16, folds=4,
    fold_seed=20261010, search_seed=871, search_fit_seed=101,
    depth_min=3, depth_max=8, widths=(16, 32, 64, 128),
    shared_layers_max=3, private_layers_max=4, strength_max=100000.0,
)
BASELINE = Architecture(depth=3, shared_width=32, shared_layers=0,
    private_width=32, private_layers=3, regularization_strength=500.0, x_dim=6)
TRAINING = TrainingConfig(epochs=200, batch_size=256, lr=1e-3, patience=50,
    validation_every=5, min_delta=1e-4, max_grad_norm=1.0, cuda_graphs=True)
OPTIMIZATION = NLLCVOptimization(split_lr_ratio=0.1, warmup_epochs=10)

torch.set_num_threads(CPU_THREADS)
compatibility = verify_generator()  # raises BEFORE any fit if data generators differ
KWARGS = dict(config=CONFIG, search=SEARCH, training=TRAINING,
              optimization=OPTIMIZATION, baseline=BASELINE, device=DEVICE)
n_settings = len(expanded_settings(CONFIG))
print('Generator identity:', compatibility['passed'])
print('Device:', DEVICE, 'Output:', OUTPUT, 'Methods:', METHODS)
print('Settings:', n_settings)
print('CV fits:', n_settings*(2*SEARCH.trials+1)*SEARCH.folds)
print('Final fits:', n_settings*len(METHODS)*len(CONFIG.seeds))

## Tune on the first 80% only

四折验证 NLL 选择 checkpoint、候选和最终参数。中断后以同一配置恢复缓存。

In [ ]:
selected, trials, folds = tune_nll_cv(OUTPUT, **KWARGS)
print('Selected settings:', selected.setting.nunique(), 'CV fits:', len(folds))
display(selected[['setting', 'method', 'depth', 'shared_layers', 'shared_width',
    'private_layers', 'private_width', 'regularization_strength', 'refit_epochs', 'cv_nll']])

## Refit frozen selections and evaluate final seeds

每个 seed 使用新生成的5000样本的前80%拟合和后20%测试。测试数据不选择停止轮数。

In [ ]:
runs, summary = evaluate_nll_cv(OUTPUT, **KWARGS)
print('Final fits:', len(runs))
display(summary)

In [ ]:
from pycode.unified_nll.raw_time.report import report_nll_cv, audit_nll_cv, plot_nll_cv
import matplotlib.pyplot as plt
overall, family, paired = report_nll_cv(OUTPUT)
display(overall)
display(family)
display(paired)
print(audit_nll_cv(OUTPUT))
plot_nll_cv(OUTPUT)
plt.show()